# 🔌 InfoDiet — Browser Extension Playground

Install the **InfoDiet browser extension**, connect it to a live copy of the app running from this
notebook, and watch a real article you read travel the entire pipeline — with a live status
dashboard, one-click inspectors, and an automated PASS / WAIT / FAIL experiment kit.

## 🚀 5-minute Quick Start

| | Step | Where |
|---|---|---|
| 1 | **Run Setup** | Section 1 (one cell) |
| 2 | **Run Launch** | Section 2 — prints your app's public URL |
| 3 | **Install the extension** | Section 4 — `chrome://extensions` → Load unpacked |
| 4 | **Copy the token** | Section 5 → paste into the extension's Options → Save |
| 5 | **Open a news article** | any BBC / NPR / Reuters *article* page — the badge flashes ✓ |
| 6 | **Watch it propagate** | Section 8 — the live dashboard turns 🟢 row by row |

Sections 9–12 are the detailed playground: one-click inspectors, the full experiment kit,
a debugging guide, and a reset cell for repeat experiments.

## What's happening?

```
  Browser — you read a news article
     ↓
  Extension — URL + standard page metadata only, never article text
     ↓  POST /api/me/reads · Authorization: Bearer <your token>
  Web tier — resolves the token to your account
     ↓
  Engine — scores the read → your Reading History
     ↓
  FeedEntry → FeedArticle — born "provisional" in the shared catalog
     ↓
  Search + Stories — findable immediately
     ↓
  Refresh cycle — the next poller cycle picks the catalog change up
     ↓
  Recommendation graph — other readers can now be recommended it
     ↓
  Discover — after promotion to "verified" (a feed re-finds it, or a 2nd reader reads it)
```

**Privacy:** the extension records only the URL and standard page metadata (title, description,
image link, publisher, publication date) of supported news articles — never article text, browsing
history, cookies, or anything on non-supported sites. Section 3 is the full explainer.


In [ ]:
#@title 1 · Setup — clone the repo (skipped if you're already inside it) + Node 20
REPO   = "greenwichg/random_walks_with_erasure"  #@param {type:"string"}
BRANCH = "claude/sleepy-gates-oecof1"             #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}   # only needed if the repo is private

import pathlib, subprocess
if pathlib.Path("examples/api_fastapi.py").exists():
    print("✅ Already inside the repo — clone skipped:", pathlib.Path.cwd())
else:
    auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
    %cd /content
    !rm -rf app && git clone --depth 1 --branch {BRANCH} https://{auth}github.com/{REPO}.git app
    %cd app

# Colab ships an old Node; Next.js 14 needs >= 18 (installed once — re-runs skip it).
need_node = True
try:
    v = subprocess.run(["node", "-v"], capture_output=True, text=True).stdout.strip()
    need_node = int(v.lstrip("v").split(".")[0]) < 18
except Exception:
    pass
if need_node:
    !curl -fsSL https://deb.nodesource.com/setup_20.x | sudo -E bash - >/dev/null 2>&1
    !sudo apt-get install -y nodejs >/dev/null 2>&1
!node -v ; npm -v


In [ ]:
#@title 2 · Launch — engine + web app + public tunnel (safe to re-run)
#@markdown **live-feed** ingests real cross-spectrum RSS so recommendations carry real, openable
#@markdown URLs — what this playground is designed around. **synthetic** boots offline, but the
#@markdown dashboard's refresh/recommendation rows and kit stages 6/8/9 need live-feed.
CORPUS = "live-feed"  #@param ["live-feed", "synthetic"]
#@markdown How often the engine re-polls sources and hot-refreshes the recommendation corpus —
#@markdown 60 s keeps the dashboard's "Refresh → Recommendation" wait to about a minute.
POLL_SECONDS = 60  #@param {type:"integer"}
#@markdown A Cloudflare quick tunnel gives the app a public URL (required on Colab). Untick ONLY
#@markdown when running this notebook on your own machine — the extension then uses localhost:3000.
TUNNEL = True  #@param {type:"boolean"}
FORCE_WEB_REBUILD = False  #@param {type:"boolean"}

import pathlib, subprocess, sys

# ---- the shared runtime helpers (Sections 5–12 import this; gitignored dev artifact) ----
pathlib.Path("_playground_ui.py").write_text(r'''"""Runtime helpers for deploy/browser_extension_playground.ipynb (written by its Section 2).

Pure orchestration over the existing engine/web surfaces -- no business logic lives here.
Gitignored dev artifact: safe to delete; re-running Section 2 recreates it. State lives in
playground_state.json so every section keeps working after a kernel restart (Colab keeps the
engine/web/tunnel processes alive across restarts).
"""
import json
import os
import pathlib
import re
import secrets
import subprocess
import sys
import time
import urllib.request

_HERE = pathlib.Path(__file__).resolve().parent
sys.path.insert(0, str(_HERE / "examples"))      # the engine's own modules (ingest, store)

STATE_FILE = _HERE / "playground_state.json"
ENGINE = "http://127.0.0.1:8000"
WEB_LOCAL = "http://127.0.0.1:3000"


# ---------- state ----------
def load_state():
    try:
        return json.loads(STATE_FILE.read_text())
    except Exception:
        return {}


def save_state(**kv):
    st = load_state()
    st.update(kv)
    STATE_FILE.write_text(json.dumps(st, indent=1))
    return st


# ---------- tiny HTTP ----------
def req_json(method, url, body=None, headers=None, timeout=30):
    data = json.dumps(body).encode() if body is not None else None
    r = urllib.request.Request(url, data=data, method=method,
                               headers={"Content-Type": "application/json", **(headers or {})})
    with urllib.request.urlopen(r, timeout=timeout) as resp:
        return json.loads(resp.read().decode() or "{}")


def get(path, uid=None, base=ENGINE):
    return req_json("GET", base + path, headers={"X-IH-User-Id": str(uid)} if uid else {})


def post(path, body, uid=None, base=ENGINE):
    return req_json("POST", base + path, body, {"X-IH-User-Id": str(uid)} if uid else {})


def ping(url, timeout=3):
    try:
        return urllib.request.urlopen(url, timeout=timeout).status == 200
    except Exception:
        return False


def wait_http(url, seconds=240):
    for _ in range(seconds):
        if ping(url, 2):
            return True
        time.sleep(1)
    return False


# ---------- colored status blocks (HTML in Colab; plain text anywhere else) ----------
_COLORS = {"ok": ("#e6f4ea", "#137333", "\U0001f7e2"),
           "wait": ("#fef7e0", "#b06000", "\U0001f7e1"),
           "fail": ("#fce8e6", "#c5221f", "\U0001f534"),
           "skip": ("#f1f3f4", "#5f6368", "\u26aa")}


def block(state, title, detail=""):
    """One colored status row; returns True for "ok" so callers can chain verdicts."""
    bg, fg, dot = _COLORS[state]
    html = (f'<div style="background:{bg};color:{fg};border-left:6px solid {fg};'
            f'border-radius:6px;padding:9px 14px;margin:5px 0;font-family:monospace;'
            f'font-size:13px"><b>{dot} {title}</b>'
            + (f'<div style="margin-top:3px;color:#3c4043">{detail}</div>' if detail else "")
            + "</div>")
    try:
        from IPython.display import display, HTML
        display(HTML(html))
    except Exception:
        print(f"{dot} {title}" + (f" -- {detail}" if detail else ""))
    return state == "ok"


def block_rows(rows, footer=""):
    for state, title, detail in rows:
        block(state, title, detail)
    if footer:
        print(footer)


def clear():
    try:
        from IPython.display import clear_output
        clear_output(wait=True)
    except Exception:
        pass


# ---------- engine-native joins (the same canonicalizer + store the engine uses) ----------
_STORE = None


def _store():
    global _STORE
    if _STORE is None:
        import store
        _STORE = store.Store()
    return _STORE


def canon(url):
    import ingest
    return ingest.canonical_url(url)


def slug_of(canonical):
    import urllib.parse
    path = urllib.parse.urlparse(canonical).path.rstrip("/")
    return path.rsplit("/", 1)[-1] or canonical


def feed_row(url):
    """The FeedArticle row for a URL -- read straight from the store because the lifecycle
    field (articleState) is deliberately not public API (same pattern as the experiment kit)."""
    return _store().get_feed_article(canon(url))


def clear_reads(user_id):
    """Developer reset: delete one reader's read rows via the engine's own ORM. Restart the
    engine afterwards so its in-memory caches reload (Section 12 enforces that)."""
    import store
    ses = _store()._Session()
    try:
        n = ses.query(store.Read).filter(store.Read.user_id == int(user_id)).delete()
        ses.commit()
        return n
    finally:
        ses.close()


# ---------- services (Section 2 launches; Section 12 restarts) ----------
def pkill(pattern):
    subprocess.run(["pkill", "-f", pattern], stderr=subprocess.DEVNULL)


def start_engine(env_overrides):
    """(Re)start the FastAPI engine with the playground env; True once /api/health responds.
    Kills any stale engine first -- Colab keeps old subprocesses alive across cell re-runs."""
    pkill("examples/api_fastapi.py")
    time.sleep(2)
    env = {**os.environ, **{k: str(v) for k, v in (env_overrides or {}).items()}}
    subprocess.Popen([sys.executable, "examples/api_fastapi.py"], env=env, cwd=str(_HERE),
                     stdout=open(_HERE / "engine.log", "w"), stderr=subprocess.STDOUT)
    return wait_http(ENGINE + "/api/health")


def write_web_env(public_url=None):
    lines = ["RWE_BACKEND_URL=http://127.0.0.1:8000",
             f"NEXTAUTH_SECRET={secrets.token_urlsafe(32)}",
             "RWE_DEV_LOGIN=1", "NEXT_PUBLIC_DEV_LOGIN=1"]
    if public_url:
        lines.insert(2, f"NEXTAUTH_URL={public_url}")
    (_HERE / "web" / ".env.local").write_text("\n".join(lines) + "\n")


def start_web():
    pkill("next-server")
    time.sleep(2)
    subprocess.Popen(["npm", "start"], cwd=str(_HERE / "web"),
                     stdout=open(_HERE / "web.log", "w"), stderr=subprocess.STDOUT)
    return wait_http(WEB_LOCAL + "/onboarding", 180)


def start_tunnel():
    """Cloudflare quick tunnel to the WEB app -- the public URL the extension must be given
    (the engine stays private on localhost). Returns the URL, or None (just retry)."""
    pkill("cloudflared")
    time.sleep(1)
    binp = _HERE / "cloudflared"
    if not binp.exists() or binp.stat().st_size < 1_000_000:
        subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/"
                       "download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared",
                       shell=True, cwd=str(_HERE))
    (_HERE / "cf.log").write_text("")
    subprocess.Popen([str(binp), "tunnel", "--url", WEB_LOCAL, "--no-autoupdate"],
                     stdout=open(_HERE / "cf.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(90):
        time.sleep(1)
        m = re.search(r"https://[-\w.]+\.trycloudflare\.com", (_HERE / "cf.log").read_text())
        if m:
            return m.group(0)
    return None
''')
import importlib, _playground_ui as ui
ui = importlib.reload(ui)

# ---- 2a · engine — feed mode + background poller (the refresh loop the pipeline needs) ----
try:
    import fastapi, uvicorn, sqlalchemy  # noqa: F401 — the [serve] extra, already installed
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[serve]"])

eng_env = {}
if CORPUS == "live-feed":
    subprocess.run([sys.executable, "examples/rss_ingest.py", "run",
                    "--feeds", "deploy/rss_feeds.example.txt"])
    subprocess.run([sys.executable, "examples/rss_ingest.py", "status"])
    eng_env = {"RWE_RECS_SOURCE": "feed", "RWE_FEED_MAX_PER_OUTLET": "40",
               # the poller only runs cycles when it has a feeds spec — and a cycle must run for
               # the after-your-read refresh check (catalogDirty) to fire
               "RWE_RSS_FEEDS": "deploy/rss_feeds.example.txt",
               "RWE_FEED_POLL": "1", "RWE_POLL_INTERVAL": str(POLL_SECONDS)}

up = ui.start_engine(eng_env)
src = {}
if up:
    try:
        src = ui.get("/api/health").get("recommendationSource") or {}
    except Exception:
        pass
ui.block("ok" if up else "fail", "Engine Running" if up else "Engine FAILED",
         (f"corpus: {src.get('source', '?')} · feed articles: {src.get('feedArticles', '?')} · "
          f"poll every {POLL_SECONDS}s") if up else "engine.log tail printed below")
if not up:
    print("\n".join(open("engine.log").read().splitlines()[-15:]))

# ---- 2b · web app — production build (dev mode breaks behind a tunnel); demo sign-in on ----
ui.write_web_env()
if pathlib.Path("web/.next/BUILD_ID").exists() and not FORCE_WEB_REBUILD:
    print("web build found — reusing it (tick FORCE_WEB_REBUILD to rebuild)")
else:
    subprocess.run(["npm", "install", "--no-audit", "--no-fund", "--loglevel=error"], cwd="web")
    subprocess.run(["npm", "run", "build"], cwd="web")
web_up = ui.start_web()
ui.block("ok" if web_up else "fail", "Web Running" if web_up else "Web FAILED",
         "" if web_up else "web.log has the details")

# ---- 2c · the public URL the extension will be pointed at ----
PUBLIC_URL = None
if web_up and TUNNEL:
    PUBLIC_URL = ui.start_tunnel()
    if PUBLIC_URL:
        ui.write_web_env(PUBLIC_URL)   # sign-in redirects must target the public URL
        ui.start_web()                 # runtime env only — the build (and demo login) stays
elif web_up:
    PUBLIC_URL = "http://localhost:3000"

ui.save_state(corpus=CORPUS, pollSeconds=POLL_SECONDS, engineEnv=eng_env)
if PUBLIC_URL:
    ui.save_state(publicUrl=PUBLIC_URL)     # kept out of the failure path so a good URL survives
    if TUNNEL:
        ui.block("ok", "Tunnel Running",
                 f'app URL: <a href="{PUBLIC_URL}" target="_blank"><b>{PUBLIC_URL}</b></a> — the '
                 'extension\'s "InfoDiet app URL" must be exactly this')
    else:
        ui.block("ok", "Local mode (no tunnel)", "app URL for the extension: http://localhost:3000")
else:
    ui.block("fail", "Tunnel not up", "quick tunnels are sometimes slow/rate-limited — just "
             "re-run this cell (it usually works on the 2nd try); cf.log tail below")
    print("    " + "\n    ".join(open("cf.log").read().splitlines()[-10:] or ["(empty)"]))

if up and web_up and PUBLIC_URL:
    print("\n👉 Open the app:", PUBLIC_URL)
    print('   Sign in → "Continue as demo reader".')
    print("   Next: Sections 3–4 (install the extension) → 5 (token) → 6 (connection test).")


## 3 · What the extension is (and is not)

A deliberately tiny Manifest V3 extension. On **supported news sites only** (a static allowlist in
its `manifest.json`), it detects when the page is an *article* (OpenGraph / JSON-LD / `<article>`
markup) and records the read to your InfoDiet account.

**It collects, per article you open:** the URL, title, description, image link, publisher,
publication date — the standard metadata already in the page's `<head>`.

**It never collects:** article text · browsing history · cookies · passwords · anything on
non-supported sites · anything when the page isn't an article (section fronts are ignored).

```
content.js (allowlisted news page)   → detects the article, reads standard metadata
background.js (service worker)       → de-dups locally (6 h), POSTs to YOUR app URL
web tier                             → resolves your Bearer token → forwards to the engine
engine                               → scores it → Reading History + a provisional FeedArticle
```

The extension talks **only to your InfoDiet web app URL** (the tunnel URL from Section 2) — never
to the engine, never to third parties. The token lives only in your browser; Section 12 (or the
app's Settings) revokes it any time.

**Badge legend** — what the toolbar icon tells you after you open an article:

| Badge | Meaning |
|---|---|
| ✓ (green flash) | read recorded |
| `auth` | token missing/revoked → regenerate in Section 5, re-save Options |
| `err` | app URL wrong or tunnel dead → re-run Section 2, update Options |
| *no badge at all* | not detected: section page, or the site isn't on the allowlist (by design) |


## 4 · Install the extension (about 2 minutes, Chrome / Edge / Brave)

The extension is the `extension/` folder of this repo. Get it onto the machine whose browser
you'll read with (your laptop — not the Colab VM):

- **ZIP**: download <https://github.com/greenwichg/random_walks_with_erasure/archive/refs/heads/claude/sleepy-gates-oecof1.zip>,
  unzip, and find the `extension/` folder inside, **or**
- **git**: `git clone -b claude/sleepy-gates-oecof1 https://github.com/greenwichg/random_walks_with_erasure.git`

Then:

1. Open `chrome://extensions` and switch **Developer mode** ON (top-right toggle).
2. Click **Load unpacked** → select the `extension/` folder.
3. Right-click the InfoDiet toolbar icon → **Options** (or `chrome://extensions` → Details →
   Extension options). Keep this page open — Section 5 gives you the two values to paste.

> Re-loading after an update: `chrome://extensions` → the ↻ reload icon on the InfoDiet card.


In [ ]:
#@title 5 · Generate your extension token (shown once — the engine stores only its hash)
LABEL = "colab-playground"  #@param {type:"string"}
REGENERATE = False  #@param {type:"boolean"}

import _playground_ui as ui

st = ui.load_state()
app_url = st.get("publicUrl") or "http://localhost:3000"

# The SAME account the "Continue as demo reader" button signs into (upsert by identity) — so
# extension reads land on the account you see in the app.
uid = ui.post("/api/internal/users",
              {"provider": "dev", "providerAccountId": "demo@infodiet.local",
               "email": "demo@infodiet.local", "displayName": "Demo Reader"})["userId"]
mine = [t for t in ui.get("/api/me/tokens", uid) if t.get("label") == LABEL]
token = st.get("token")

if REGENERATE or not token or not mine:   # not mine = fresh DB → the saved plaintext is stale
    for t in mine:                         # revoke stale playground tokens first
        ui.req_json("DELETE", ui.ENGINE + f"/api/me/tokens/{t['id']}",
                    headers={"X-IH-User-Id": str(uid)})
    minted = ui.post("/api/me/tokens", {"label": LABEL}, uid)
    token = minted["token"]
    ui.save_state(token=token, tokenId=minted["id"], tokenLabel=LABEL, uid=uid)
    ui.block("ok", "Token Generated", f'label "{LABEL}" · token id {minted["id"]}')
else:
    ui.save_state(uid=uid, tokenLabel=LABEL)
    ui.block("ok", "Token Ready", "reusing this playground's token (tick REGENERATE for a fresh one)")

print("\nPaste these into the extension → right-click the icon → Options:")
print("  InfoDiet app URL :", app_url)
print("  API token        :", token)
print("\nThen click Save (grants the extension access to that origin) → Test connection.")
print("Revoke any time: Section 12, or delete it in the app's Settings.")


In [ ]:
#@title 6 · Connection test — the exact request the extension's "Test connection" sends
import json, urllib.error, urllib.request
import _playground_ui as ui

st = ui.load_state()
app_url, token = st.get("publicUrl") or "http://localhost:3000", st.get("token")

chain = True
try:
    ui.get("/api/health")
    ui.block("ok", "Engine reachable", "localhost:8000 (stays private — the extension never sees it)")
except Exception as e:
    chain = ui.block("fail", "Engine not running", f"{e} — re-run Section 2")
if ui.ping(ui.WEB_LOCAL + "/onboarding"):
    ui.block("ok", "Web app running", "localhost:3000")
else:
    chain = ui.block("fail", "Web app not running", "re-run Section 2 (web.log has details)")

if not token:
    ui.block("fail", "No token yet", "run Section 5 first")
elif chain:
    # An EMPTY reads batch with the Bearer token — byte-for-byte what the extension's own
    # "Test connection" button sends. Green here = the real extension will connect too.
    req = urllib.request.Request(app_url + "/api/me/reads",
                                 data=json.dumps({"reads": []}).encode(),
                                 headers={"Content-Type": "application/json",
                                          "Authorization": f"Bearer {token}"})
    try:
        body = json.loads(urllib.request.urlopen(req, timeout=20).read().decode())
        ui.block("ok", "Extension Connected (simulated)",
                 f"token accepted at {app_url} · accepted={body.get('accepted')} — paste the same "
                 "URL + token into the extension Options and its test will match")
    except urllib.error.HTTPError as e:
        if e.code == 401:
            ui.block("fail", "Auth Failed (401)",
                     "token unknown or revoked — tick REGENERATE in Section 5, then re-save the "
                     "extension Options with the new token")
        else:
            ui.block("fail", f"App Not Healthy (HTTP {e.code})",
                     "the URL answers but the app errored — check web.log / engine.log")
    except Exception as e:
        ui.block("fail", "Unreachable",
                 f"{app_url} — the tunnel died or its URL rotated: re-run Section 2, update the "
                 f"extension Options ({e})")


## 7 · Read a real article (the only step that needs your hands)

Open an **article page** on a supported outlet in the browser with the extension installed —
the toolbar badge flashes **✓** when the read is recorded.

**Supported outlets** (the extension's static allowlist): NYT · Washington Post · WSJ · Fox News ·
CNN · NBC · AP · Reuters · NPR · BBC · The Guardian · Politico · The Hill · USA Today · CNBC ·
Bloomberg · ABC · CBS · Al Jazeera · Axios · Vox · The Atlantic · LA Times.

Easy first picks (metadata-rich, no paywall): **BBC, NPR, Reuters, AP, The Guardian, Al Jazeera, Axios**.

Two rules of thumb:

- An **article** page, not a section front — `bbc.com/news/articles/…` works,
  `bbc.com/news` does nothing (by design: the detector only fires on articles).
- A **fresh** article you haven't opened in the last 6 hours (the extension de-dups locally).

Then copy the article's URL and head to Section 8 — the dashboard watches it travel the pipeline.


In [ ]:
#@title 8 · Live validation dashboard — watch your read travel the pipeline
#@markdown Paste the article URL you opened with the extension (blank = just watch for ANY
#@markdown extension read). The dashboard refreshes itself for WATCH_SECONDS (0 = one snapshot);
#@markdown interrupt the cell (⏹) to stop early — re-run to watch again.
ARTICLE_URL = ""  #@param {type:"string"}
WATCH_SECONDS = 180  #@param {type:"integer"}
REFRESH_EVERY = 5  #@param {type:"integer"}

import json, time, urllib.parse
import _playground_ui as ui

if ARTICLE_URL and ARTICLE_URL != ui.load_state().get("articleUrl"):
    ui.save_state(articleUrl=ARTICLE_URL, baselineGeneration=None)   # new article → new watch
target = ARTICLE_URL or ui.load_state().get("articleUrl") or ""
canon = ui.canon(target) if target else ""
slug = ui.slug_of(canon) if target else ""


def rows():
    st = ui.load_state()
    uid = st.get("uid") or 1
    R = []
    # 1 · App Running
    eng = src = None
    try:
        h = ui.get("/api/health")
        eng = True
        src = (h.get("recommendationSource") or {}).get("source", "?")
    except Exception:
        eng = False
    web = ui.ping(ui.WEB_LOCAL + "/onboarding")
    R.append(("ok" if eng and web else "fail", "App Running",
              f"engine {'✓' if eng else '✗'} · web {'✓' if web else '✗'} · corpus: {src}"
              + ("" if eng and web else " — re-run Section 2")))
    if not eng:
        return R
    try:
        hist = ui.get("/api/me/history", uid)
    except Exception:
        hist = []
    ext = [e for e in hist if e.get("readSource") == "extension"]
    # 2 · Extension Connected — proven end-to-end by an extension read landing in history
    if ext:
        R.append(("ok", "Extension Connected",
                  f"{len(ext)} extension read(s) — latest: "
                  f"{((ext[0].get('article') or {}).get('headline') or '?')[:60]}"))
    elif st.get("token"):
        R.append(("wait", "Extension Connected",
                  "token ready — waiting for your first extension read (badge ✓); "
                  "Section 6 tests the chain without a browser"))
    else:
        R.append(("fail", "Extension Connected", "no token — run Section 5"))
    # 3 · Read Recorded
    if target:
        mine = [e for e in hist if slug and slug in ((e.get("article") or {}).get("url") or "")]
        if mine:
            R.append(("ok", "Read Recorded",
                      f"in your history · readSource={mine[0].get('readSource')} · "
                      f"{mine[0].get('readAt') or ''}"))
        else:
            R.append(("wait", "Read Recorded",
                      "URL not in history yet — did the badge flash ✓? (section pages and "
                      "non-allowlisted sites are ignored by design)"))
    else:
        R.append((("ok" if ext else "wait"), "Read Recorded",
                  f"{len(ext)} extension read(s) so far" if ext
                  else "set ARTICLE_URL above to track one article precisely"))
    if not target:
        for name in ("FeedArticle Created", "Search Indexed", "Stories Updated",
                     "Recommendation Ready", "Discover Promotion"):
            R.append(("skip", name, "set ARTICLE_URL to track a specific article"))
        return R
    # 4 · FeedArticle Created (direct store read — lifecycle isn't public API, same as the kit)
    row = ui.feed_row(target)
    if row:
        R.append(("ok", "FeedArticle Created",
                  f"sourceType={row.get('sourceType')} · articleState={row.get('articleState')} · "
                  f"lean={(row.get('scored') or {}).get('lean')}"))
    else:
        R.append(("wait", "FeedArticle Created",
                  "no catalog row yet — it arrives with the read; if Read Recorded is 🟢 but this "
                  "stays 🟡, grep engine.log for extension_catalog_failed"))
    # 5 · Search Indexed
    words = " ".join((((row or {}).get("title") or slug.replace("-", " ")).split())[:3])
    hit = False
    if words:
        try:
            res = ui.get("/api/search?query=" + urllib.parse.quote(words))
            hit = any(slug in (a.get("url") or "") for a in res.get("results", []))
        except Exception:
            pass
    R.append(("ok" if hit else "wait", "Search Indexed",
              f'query "{words}" finds it' if hit
              else f'query "{words}" — not yet (instant once the catalog row exists)'))
    # 6 · Stories Updated
    clustered = False
    try:
        sres = ui.get("/api/stories?sort=latest&limit=50")
        clustered = slug in json.dumps(sres.get("stories", sres if isinstance(sres, list) else []))
    except Exception:
        pass
    R.append(("ok" if clustered else "wait", "Stories Updated",
              "clustered into a story" if clustered
              else "needs the same event from a 2nd outlet (≥2 articles · ≥2 publishers) — "
                   "a 🟡 here is the expected state"))
    # 7 · Recommendation Ready — the corpus refresh has picked your article up
    lean = ((row or {}).get("scored") or {}).get("lean") if row else None
    try:
        ref = ui.get("/api/internal/refresh")
    except Exception:
        ref = {}
    gen, dirty = ref.get("generation"), ref.get("catalogDirty")
    base = st.get("baselineGeneration")
    if row and base is None and gen is not None:
        st = ui.save_state(baselineGeneration=gen)    # first sight of the row = the baseline
        base = gen
    if not (st.get("engineEnv") or {}).get("RWE_FEED_POLL"):
        R.append(("wait", "Recommendation Ready",
                  "no refresh loop in synthetic mode — relaunch Section 2 with CORPUS=live-feed"))
    elif row and lean is None:
        R.append(("fail", "Recommendation Ready",
                  "lean unresolved — this outlet can't join the recommendation graph (the "
                  "documented unknown-outlet gap); pick a major allowlisted outlet"))
    elif row and gen is not None and base is not None and gen > base:
        R.append(("ok", "Recommendation Ready",
                  f"corpus refreshed past your read (generation {base} → {gen}) — other readers "
                  "can now receive it; prove it with Section 10 (stage 8)"))
    else:
        R.append(("wait", "Recommendation Ready",
                  f"waiting for the next refresh cycle (≤ {st.get('pollSeconds', '?')}s) · "
                  f"catalogDirty={dirty} · generation={gen}"))
    # 8 · Discover Promotion
    astate = (row or {}).get("articleState")
    vis = False
    try:
        vis = any(slug in (a.get("url") or "")
                  for a in ui.get("/api/discover?limit=200").get("articles", []))
    except Exception:
        pass
    if not row:
        R.append(("wait", "Discover Promotion", "waiting for the catalog row"))
    elif astate == "provisional":
        R.append(("wait" if not vis else "fail", "Discover Promotion",
                  "provisional → hidden from Discover by design; promotes to verified when a feed "
                  "re-finds it or a 2nd reader reads it (Section 10, stage 9)" if not vis
                  else "provisional article visible in Discover — should never happen"))
    else:
        R.append(("ok" if vis else "wait", "Discover Promotion",
                  f"articleState={astate} · "
                  + ("visible in Discover" if vis else "promoted — appears in Discover shortly")))
    return R


deadline = time.time() + max(WATCH_SECONDS, 0)
try:
    while True:
        snapshot = rows()
        ui.clear()
        ui.block_rows(snapshot, footer=time.strftime("updated %H:%M:%S — ⏹ stops the watch"))
        if time.time() >= deadline:
            break
        time.sleep(max(REFRESH_EVERY, 1))
except KeyboardInterrupt:
    pass
print("watch ended — re-run this cell to keep watching.")


In [ ]:
#@title 9 · Developer Playground — one-click inspectors
#@markdown Buttons in Colab (via ipywidgets); anywhere they can't render, set ACTION and run the
#@markdown cell. TARGET_URL defaults to the Section-8 article.
TARGET_URL = ""  #@param {type:"string"}
ACTION = "All"  #@param ["All", "FeedArticle", "Story", "Recommendation", "Reader", "Refresh", "Metrics"]
USE_BUTTONS = True  #@param {type:"boolean"}

import json, urllib.parse
import _playground_ui as ui

st = ui.load_state()
uid = st.get("uid") or 1
target = TARGET_URL or st.get("articleUrl") or ""


def _p(title, obj):
    print(f"— {title} " + "—" * max(0, 66 - len(title)))
    print(json.dumps(obj, indent=1, ensure_ascii=False)[:4000])


def inspect_feed_article():
    if not target:
        return print("set TARGET_URL (or run Section 8 with ARTICLE_URL first)")
    row = ui.feed_row(target)
    if not row:
        return print("no FeedArticle for", ui.canon(target))
    keep = {k: row.get(k) for k in ("title", "publisher", "canonicalUrl", "sourceType",
                                    "articleState", "sourceProvider", "publishedAt",
                                    "fetchedAt", "image")}
    keep["scored"] = row.get("scored")
    _p("FeedArticle (store row — includes the lifecycle fields)", keep)


def inspect_story():
    res = ui.get("/api/stories?sort=latest&limit=20")
    stories = res.get("stories", res if isinstance(res, list) else [])
    if target:
        s = ui.slug_of(ui.canon(target))
        mine = [x for x in stories if s in json.dumps(x)]
        return _p(f"stories containing your article ({len(mine)})",
                  mine or ["(none yet — read the same event on a 2nd outlet)"])
    _p("latest stories", stories[:5])


def inspect_recommendation():
    got = {}
    for q in ("", "?strategy=rwe-b", "?strategy=rwe-d", "?strategy=adaptive"):
        name = q.split("=")[-1] if q else "default"
        try:
            for r in ui.get("/api/recommendations" + q, uid):
                u = (r.get("article") or {}).get("url") or str(r.get("articleId", ""))
                got.setdefault(u, []).append(name)
        except Exception as e:
            got[f"(error on {name})"] = [str(e)]
    print(f"reader #{uid} — union across strategies: {len(got)} articles")
    for u, names in list(got.items())[:15]:
        print(f"  {','.join(names):30s} {u[:88]}")
    if target:
        s = ui.slug_of(ui.canon(target))
        print("\nyour own article in YOUR feed:", any(s in u for u in got),
              "(False is correct — you read it, so seen-exclusion removes it; another reader "
              "receiving it is proven by Section 10, stage 8)")


def inspect_reader():
    try:
        hist = ui.get("/api/me/history", uid)
    except Exception as e:
        return print(f"no history for #{uid}: {e}")
    by_src = {}
    for e in hist:
        k = e.get("readSource") or "app/legacy"
        by_src[k] = by_src.get(k, 0) + 1
    print(f"reader #{uid}: {len(hist)} reads · by source: {by_src}")
    for e in hist[:8]:
        a = e.get("article") or {}
        print(f"  [{(e.get('readSource') or '-'):9s}] {(a.get('headline') or '?')[:70]}")
    print("tokens:", [(t.get("id"), t.get("label")) for t in ui.get("/api/me/tokens", uid)])


def inspect_refresh():
    try:
        _p("refresh snapshot (/api/internal/refresh)", ui.get("/api/internal/refresh"))
    except Exception as e:
        print("refresh snapshot unavailable:", e)
    try:
        lines = [l for l in open("engine.log").read().splitlines()
                 if any(k in l for k in ("source_poll", "corpus_refresh", "extension_catalog"))]
        print("\nengine.log pipeline events (last 10):")
        print("\n".join(lines[-10:]) or "(none yet — the first poll cycle hasn't run)")
    except FileNotFoundError:
        print("engine.log not found (engine started outside this notebook?)")


def inspect_metrics():
    rep = ui.get("/api/report", uid)
    print(f"Information Health — overall {rep.get('overall')} ({rep.get('band')}) · "
          f"mode={rep.get('mode')}")
    for m in rep.get("metrics", []):
        print(f"  {m.get('key', '?'):16s} score={m.get('score'):3d}  Δ{m.get('delta'):+d}  "
              f"band={m.get('band')}")
    if rep.get("coverage"):
        print("coverage:", rep["coverage"])


ACTIONS = {"FeedArticle": inspect_feed_article, "Story": inspect_story,
           "Recommendation": inspect_recommendation, "Reader": inspect_reader,
           "Refresh": inspect_refresh, "Metrics": inspect_metrics}

buttons_ok = False
if USE_BUTTONS:
    try:
        import ipywidgets as W
        from IPython.display import display

        out = W.Output()

        def _mk(name):
            b = W.Button(description=f"Inspect {name}", layout=W.Layout(width="auto"))
            def _on(_b, _name=name):
                with out:
                    out.clear_output()
                    ACTIONS[_name]()
            b.on_click(_on)
            return b

        display(W.HBox([_mk(n) for n in ACTIONS]), out)
        buttons_ok = True
    except Exception:
        pass

if not buttons_ok:
    todo = ACTIONS.items() if ACTION == "All" else [(ACTION, ACTIONS[ACTION])]
    for name, fn in todo:
        print("\n" + "=" * 74)
        print("▶", name)
        print("=" * 74)
        fn()


In [ ]:
#@title 10 · Run the automated experiment kit — PASS / WAIT / FAIL per pipeline stage
#@markdown `examples/extension_experiment.py` is the deep-dive validator behind
#@markdown `docs/EXTENSION_E2E_EXPERIMENT.md`. Its stage 8 proves the full value chain: it creates
#@markdown and seeds a SECOND reader, who must then receive YOUR article as a recommendation.
#@markdown For stage 8 your own reader must be connected to the catalog graph — if your history is
#@markdown only the one extension read, open the app and read a few Discover articles first.
#@markdown Tick SIMULATE_READ only when no browser/extension is available (it submits the
#@markdown extension-shaped read itself — the stand-in for Stage 1).
KIT_URL = ""  #@param {type:"string"}
STAGE = "all"  #@param ["all", "2", "3", "4", "5", "6", "8", "9", "10"]
SIMULATE_READ = False  #@param {type:"boolean"}

import subprocess, sys
import _playground_ui as ui

st = ui.load_state()
url = KIT_URL or st.get("articleUrl") or ""
if not url:
    print("Set KIT_URL (or run Section 8 with ARTICLE_URL first).")
else:
    cmd = [sys.executable, "examples/extension_experiment.py",
           "--url", url, "--user", str(st.get("uid") or 1)]
    if STAGE != "all":
        cmd += ["--stage", STAGE]
    if SIMULATE_READ:
        cmd += ["--simulate-read"]
    print("$", " ".join(cmd), "\n")
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()


## 11 · Debugging guide

| Symptom | Likely cause | Fix |
|---|---|---|
| Badge never appears | section page / site not allowlisted / extension not loaded | open an *article* page on a supported outlet (Section 7); reload the extension |
| Badge `auth` | token revoked or mistyped | Section 5 with REGENERATE ✓ → re-save Options |
| Badge `err` | app URL wrong, or the tunnel died/rotated | re-run Section 2, paste the NEW URL into Options, re-test with Section 6 |
| Section 6 → Auth Failed | same as `auth` above | regenerate + re-save |
| Section 6 → Unreachable | tunnel down (Colab idled, runtime recycled) | re-run Section 2 (services restart; the URL changes!) |
| Read 🟢 but FeedArticle 🟡 | producer exception | `grep extension_catalog_failed engine.log` |
| Recommendation Ready stuck 🟡 | poller has no feeds spec, or no cycle yet | Section 2 sets `RWE_RSS_FEEDS`; check `grep source_poll engine.log` and `/api/internal/refresh` (Section 9 → Inspect Refresh) |
| Recommendation Ready 🔴 lean unresolved | outlet unknown to the classifier | the documented unknown-outlet gap — use a major allowlisted outlet |
| Article in Discover "too early" | it pre-existed via RSS (born active, not provisional) | Section 9 → Inspect FeedArticle: `sourceType` will be a feed type |
| Kit stage 8 stays WAIT | your reader shares no reads with Reader B (fresh account) | read a few Discover articles in the app, wait one refresh, re-run |
| Stories stuck 🟡 | only one outlet has the event | read the same story on a 2nd outlet (clustering needs ≥2 articles, ≥2 publishers) |

**Logs** (repo root on the Colab VM): `engine.log` (pipeline events: `source_poll`,
`corpus_refresh_activated`, `extension_catalog_failed`) · `web.log` · `cf.log` (tunnel).
Section 9's **Inspect Refresh** button prints the interesting engine.log lines for you.

**Colab realities:** a recycled runtime keeps nothing — re-run Sections 1–2 and update the
extension's Options with the new tunnel URL. Quick-tunnel URLs also rotate every time the tunnel
restarts.


In [ ]:
#@title 12 · Reset the playground (repeat experiments cleanly) — or full shutdown
#@markdown Defaults reset the experiment (revoke the token, restart the engine) while keeping the
#@markdown web build and tunnel URL. CLEAR_DEMO_READS empties the demo reader's history (their
#@markdown FeedArticles stay in the shared catalog). RESTART_TUNNEL rotates the public URL — the
#@markdown extension Options must be updated afterwards, so leave it off unless the tunnel died.
REVOKE_TOKEN = True  #@param {type:"boolean"}
CLEAR_DEMO_READS = False  #@param {type:"boolean"}
RESTART_ENGINE = True  #@param {type:"boolean"}
RESTART_WEB = False  #@param {type:"boolean"}
RESTART_TUNNEL = False  #@param {type:"boolean"}
FULL_SHUTDOWN = False  #@param {type:"boolean"}

import _playground_ui as ui

st = ui.load_state()
uid = st.get("uid")

if FULL_SHUTDOWN:
    if REVOKE_TOKEN and uid:
        try:
            for t in ui.get("/api/me/tokens", uid):
                if t.get("label") == (st.get("tokenLabel") or "colab-playground"):
                    ui.req_json("DELETE", ui.ENGINE + f"/api/me/tokens/{t['id']}",
                                headers={"X-IH-User-Id": str(uid)})
            ui.save_state(token=None, tokenId=None)
            ui.block("ok", "Token revoked")
        except Exception as e:
            ui.block("skip", "Token revoke skipped", f"engine already down? ({e})")
    for pat, name in (("cloudflared", "tunnel"), ("next-server", "web"),
                      ("examples/api_fastapi.py", "engine")):
        ui.pkill(pat)
        ui.block("ok", f"Stopped: {name}")
    print("\nPlayground stopped. Re-run Section 2 to start again (new tunnel URL).")
else:
    # 1 · token
    if REVOKE_TOKEN and uid:
        try:
            for t in ui.get("/api/me/tokens", uid):
                if t.get("label") == (st.get("tokenLabel") or "colab-playground"):
                    ui.req_json("DELETE", ui.ENGINE + f"/api/me/tokens/{t['id']}",
                                headers={"X-IH-User-Id": str(uid)})
            ui.save_state(token=None, tokenId=None)
            ui.block("ok", "Token revoked",
                     "the extension badge will show `auth` until Section 5 mints a new one")
        except Exception as e:
            ui.block("fail", "Token revoke failed", str(e))
    else:
        ui.block("skip", "Token kept")
    # 2 · demo reads (direct store delete — developer reset; the engine restart below reloads caches)
    if CLEAR_DEMO_READS and uid:
        n = ui.clear_reads(uid)
        RESTART_ENGINE = True
        ui.block("ok", "Demo reads cleared",
                 f"{n} read row(s) deleted for reader #{uid} — the FeedArticles those reads "
                 "created stay in the shared catalog (engine restart forced so caches reload)")
    else:
        ui.block("skip", "Demo reads kept")
    # 3 · engine
    if RESTART_ENGINE:
        up = ui.start_engine(st.get("engineEnv") or {})
        ui.block("ok" if up else "fail", "Engine restarted" if up else "Engine failed to restart",
                 "" if up else "see engine.log")
    else:
        ui.block("skip", "Engine untouched")
    # 4 · web
    if RESTART_WEB:
        pu = st.get("publicUrl") or ""
        ui.write_web_env(pu if pu.startswith("https") else None)
        ok = ui.start_web()
        ui.block("ok" if ok else "fail", "Web restarted" if ok else "Web restart failed",
                 "" if ok else "see web.log")
    else:
        ui.block("skip", "Web untouched")
    # 5 · tunnel — a restart means a NEW public URL
    if RESTART_TUNNEL:
        url = ui.start_tunnel()
        if url:
            ui.write_web_env(url)
            ui.start_web()
            ui.save_state(publicUrl=url)
            ui.block("ok", "Tunnel restarted",
                     f"NEW app URL: {url} — update the extension Options AND re-run Section 6")
        else:
            ui.block("fail", "Tunnel failed", "re-run this cell, or Section 2")
    else:
        ui.block("skip", "Tunnel untouched", "quick-tunnel URLs rotate on restart — off unless dead")
    ui.save_state(articleUrl=None, baselineGeneration=None)   # Section 8 starts a fresh watch
    print("\nReset done — read a new article (Section 7) and watch Section 8 go 🟢 again.")
